#  ⭐ `fat_medicamentos`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

In [2]:
project_root

WindowsPath('C:/Users/janet/Documents/VigiMed/vigimed')

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [4]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2026-01-15
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2026-01-15


# 🥈 Silver

normalized data, medium quality


In [5]:
silver = bronze.copy()
silver.shape

(591457, 23)

## ✅ hash

In [6]:
from utils import build_row_hash

silver["HASH_BRONZE"] = build_row_hash(
    silver, 
    silver.columns.difference(['ATUALIZADO']).tolist(), 
    algo="sha256", 
    sep="|"
)

## ✅ Missing

In [7]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1033
NOME_MEDICAMENTO_WHODRUG                          5313
PRINCIPIOS_ATIVOS_WHODRUG                         5843
CODIGO_ATC                                        5313
DETENTOR_REGISTRO                               452994
CONCENTRACAO                                    491185
COMPONENTE_SUSPEITO                             530567
ACAO_ADOTADA                                    285907
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    571291
INDICACAO_MEDDRA                                346290
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          339119
DOSE                                            362716
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       291599
DURACAO                                         504788
INICIO_ADMINISTRACAO                            309631
FIM_ADMINISTRACAO                               422856
FORMA_FARM

In [8]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [9]:
hist_silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                            0
RELACAO_MEDICAMENTO_EVENTO                        1033
NOME_MEDICAMENTO_WHODRUG                          5313
PRINCIPIOS_ATIVOS_WHODRUG                         5843
CODIGO_ATC                                        5313
DETENTOR_REGISTRO                               452994
CONCENTRACAO                                    491185
COMPONENTE_SUSPEITO                             530567
ACAO_ADOTADA                                    285907
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO    571291
INDICACAO_MEDDRA                                346290
INDICACAO_RELATADA_NOTIFICADOR_INICIAL          339119
DOSE                                            362716
FREQUENCIA_DOSE                                   3583
POSOLOGIA                                       291599
DURACAO                                         504788
INICIO_ADMINISTRACAO                            309631
FIM_ADMINISTRACAO                               422856
FORMA_FARM

In [10]:
hist_silver.shape

(591457, 24)

## ✅ DATAS

In [11]:
colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

In [12]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   INICIO_ADMINISTRACAO  281826 non-null  object
 1   FIM_ADMINISTRACAO     168601 non-null  object
dtypes: object(2)
memory usage: 9.0+ MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,20181124,None
1,20181122,20181122
2,20181103,20181115
3,20181016,20181115
4,20181024,None


In [13]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [14]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 2 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   INICIO_ADMINISTRACAO  235655 non-null  datetime64[ns]
 1   FIM_ADMINISTRACAO     153422 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 9.0 MB


,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO
0,2018-11-24,NaT
1,2018-11-22,2018-11-22
2,2018-11-03,2018-11-15
3,2018-10-16,2018-11-15
4,2018-10-24,NaT


In [15]:
hist_silver.shape

(591457, 24)

## ✅ Mappings

- RELACAO_MED_EVENTO
- COMPONENTE_SUSPEITO
- ACAO_ADOTADA

In [16]:
# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1,
    "CONCOMITANTE": 2,
    "MEDICAMENTO NAO ADMINISTRADO": 3,
    "INTERACAO": 4,
}
# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCÍPIO ATIVO": 1, 
    "EXCIPIENTE, NÃO CLASSIFICADO": 2, 
    "SOLVENTE": 3, 
    "CORANTE": 4,
    "CONSERVANTE": 5, 
    "AGENTE FLAVORIZADOR": 6, 
    "EXCESSO PERCENTUAL": 7, 
    "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9
}

# ACAO_ADOTADA
acao_adotada_map = {
    "RETIRADA DO MEDICAMENTO": 1, 
    "SEM ALTERACAO DA DOSE": 2, 
    "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, 
    "AUMENTO DA DOSE": 5
}
mappings_config = [
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "COMPONENTE_SUSPEITO",
        "map": componente_suspeito_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    }]

In [17]:
hist_silver = apply_mappings(hist_silver, mappings_config)

In [18]:
hist_silver.shape

(591457, 30)

In [19]:
hist_silver[['ACAO_ADOTADA','ACAO_ADOTADA_VALOR','ACAO_ADOTADA_CHAVE']].value_counts(dropna=False)

ACAO_ADOTADA             ACAO_ADOTADA_VALOR       ACAO_ADOTADA_CHAVE
NaN                      DESCONHECIDO             0                     285907
Retirada do medicamento  RETIRADA DO MEDICAMENTO  1                      92706
Desconhecido             DESCONHECIDO             0                      84500
Sem alteração da dose    SEM ALTERAÇÃO DA DOSE    2                      59566
Não aplicável            NÃO APLICÁVEL            3                      55100
Redução da dose          REDUÇÃO DA DOSE          4                       8882
Aumento da dose          AUMENTO DA DOSE          5                       4796
Name: count, dtype: int64

In [20]:
hist_silver[['COMPONENTE_SUSPEITO','COMPONENTE_SUSPEITO_VALOR','COMPONENTE_SUSPEITO_CHAVE']].value_counts(dropna=False)

COMPONENTE_SUSPEITO           COMPONENTE_SUSPEITO_VALOR     COMPONENTE_SUSPEITO_CHAVE
NaN                           DESCONHECIDO                  0                            530567
Princípio Ativo               PRINCÍPIO ATIVO               1                             60624
Excipiente, não classificado  EXCIPIENTE, NÃO CLASSIFICADO  2                               131
Corante                       CORANTE                       4                                39
Conservante                   CONSERVANTE                   5                                37
Solvente                      SOLVENTE                      3                                25
Excesso percentual            EXCESSO PERCENTUAL            7                                12
Antioxidante                  ANTIOXIDANTE                  8                                 8
Agente Flavorizador           AGENTE FLAVORIZADOR           6                                 7
Estabilizante                 ESTABILIZANTE       

In [21]:
hist_silver[['RELACAO_MEDICAMENTO_EVENTO'
,'RELACAO_MEDICAMENTO_EVENTO_VALOR'
,'RELACAO_MEDICAMENTO_EVENTO_CHAVE']].value_counts(dropna=False)

RELACAO_MEDICAMENTO_EVENTO    RELACAO_MEDICAMENTO_EVENTO_VALOR  RELACAO_MEDICAMENTO_EVENTO_CHAVE
Suspeito                      SUSPEITO                          1                                   401828
Concomitante                  CONCOMITANTE                      2                                   176549
Medicamento não administrado  MEDICAMENTO NÃO ADMINISTRADO      3                                     9852
Interação                     INTERAÇÃO                         4                                     2195
NaN                           DESCONHECIDO                      0                                     1033
Name: count, dtype: int64

## ✅ DURACAO

In [22]:
from utils import normalize_duracao

In [23]:
hist_silver = normalize_duracao(hist_silver, coluna="DURACAO")

In [24]:
hist_silver[['DURACAO','DURACAO_TIPO_CHAVE','DURACAO_TIPO_VALOR','DURACAO_VALOR']].value_counts(dropna=False).head(10)

DURACAO    DURACAO_TIPO_CHAVE  DURACAO_TIPO_VALOR  DURACAO_VALOR
NaN        0                   DESCONHECIDO        NaN              504788
1 dia      4                   DIA                 1.0               22808
2 dia      4                   DIA                 2.0                4437
3 dia      4                   DIA                 3.0                3309
1 hora     3                   HORA                1.0                3255
5 dia      4                   DIA                 5.0                2666
1 minuto   2                   MINUTO              1.0                2450
4 dia      4                   DIA                 4.0                2382
30 minuto  2                   MINUTO              30.0               2355
7 dia      4                   DIA                 7.0                2011
Name: count, dtype: int64

In [25]:
hist_silver.shape

(591457, 33)

## ✅ CONCENTRACAO

In [26]:
from utils.med_normalize_concentracao import normalize_concentracao


In [27]:
hist_silver = normalize_concentracao(hist_silver, col="CONCENTRACAO")

In [28]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

In [29]:
hist_silver.query("CONCENTRACAO_TIPO_CHAVE!='desconhecido'")[['CONCENTRACAO',
       'CONCENTRACAO_TIPO_CHAVE',
       'CONCENTRACAO_TIPO_VALOR', 
       'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 
       'CONCENTRACAO_VALOR_DENOMINADOR']].value_counts(dropna=False).head(10)

CONCENTRACAO  CONCENTRACAO_TIPO_CHAVE  CONCENTRACAO_TIPO_VALOR  CONCENTRACAO_VALOR  CONCENTRACAO_VALOR_NUMERADOR  CONCENTRACAO_VALOR_DENOMINADOR
NaN           0                        desconhecido             NaN                 NaN                           NaN                               491185
100mg         1                        milligram (mg)           100.0               100.0                         NaN                                 2557
500mg         1                        milligram (mg)           500.0               500.0                         NaN                                 2279
1g            2                        gram (g)                 1.0                 1.0                           NaN                                 1989
500 mg        1                        milligram (mg)           500.0               500.0                         NaN                                 1510
100 mg        1                        milligram (mg)           100.0           

In [30]:
hist_silver.CONCENTRACAO_TIPO_VALOR.value_counts(dropna=False)

CONCENTRACAO_TIPO_VALOR
desconhecido                        499050
milligram (mg)                       46849
milligram per millilitre (mg/mL)     31856
gram (g)                              7076
percent (%)                           2610
millilitre (mL)                       2192
microgram (ug)                         740
gram per millilitre (g/mL)             657
milligram per milligram (mg/mg)        362
milligram per gram (mg/g)               65
Name: count, dtype: int64

In [31]:
hist_silver.CONCENTRACAO_TIPO_CHAVE.value_counts(dropna=False)

CONCENTRACAO_TIPO_CHAVE
0    499050
1     46849
6     31856
2      7076
5      2610
4      2192
3       740
9       657
7       362
8        65
Name: count, dtype: Int64

## ✅ DOSE

In [32]:
from utils.med_normalize_dose import normalize_dose

In [33]:
hist_silver = normalize_dose(hist_silver, col="DOSE")

In [34]:
hist_silver.query("DOSE_TIPO_CHAVE!='desconhecido'")[['DOSE',
'DOSE_TIPO_CHAVE', 
'DOSE_TIPO_VALOR', 
'DOSE_VALOR']].value_counts(dropna=False).head(10)

DOSE                  DOSE_TIPO_CHAVE  DOSE_TIPO_VALOR     DOSE_VALOR
NaN                   0                desconhecido        NaN           362716
100 milligram (mg)    1                milligram (mg)      100.0          10888
50 milligram (mg)     1                milligram (mg)      50.0           10206
10 milligram (mg)     1                milligram (mg)      10.0            8807
300 milligram (mg)    1                milligram (mg)      300.0           7805
20 milligram (mg)     1                milligram (mg)      20.0            7264
1 dosage form ({DF})  5                dosage form ({DF})  1.0             7168
40 milligram (mg)     1                milligram (mg)      40.0            6389
500 milligram (mg)    1                milligram (mg)      500.0           6151
1 gram (g)            3                gram (g)            1.0             5719
Name: count, dtype: int64

In [35]:
hist_silver.shape

(591457, 41)

## Fuzzy

### ✅ VIA_ADM

In [36]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy


In [37]:
hist_silver["VIA_ADMINISTRACAO_CHAVE"] = (
    hist_silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(
        txt,
        score_threshold=80,
        return_numeric=True,
    ))
)

hist_silver["VIA_ADMINISTRACAO_MAE_PAI_CHAVE"] = (
    hist_silver["VIA_ADMINISTRACAO_MAE_PAI"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(
        txt,
        score_threshold=80,
        return_numeric=True,
    ))
)


In [38]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

In [39]:
hist_silver[["VIA_ADMINISTRACAO","VIA_ADMINISTRACAO_CHAVE"]].value_counts(dropna=False).head()

VIA_ADMINISTRACAO    VIA_ADMINISTRACAO_CHAVE
NaN                  5                          330673
oral                 5                           31428
infusão intravenosa  6                           26376
intramuscular        6                           19896
desconhecida         0                           18764
Name: count, dtype: int64

In [40]:
hist_silver[["VIA_ADMINISTRACAO_MAE_PAI","VIA_ADMINISTRACAO_MAE_PAI_CHAVE"]].value_counts(dropna=False).head()

VIA_ADMINISTRACAO_MAE_PAI   VIA_ADMINISTRACAO_MAE_PAI_CHAVE
NaN                         5                                  591302
E2B R2 Code: 050 - Other    6                                      45
Oral                        5                                      15
E2B R2 Code: 065 - Unknown  0                                      14
Unknown                     0                                      14
Name: count, dtype: int64

### ✅ DETENTOR_REGISTRO

In [41]:
hist_dim_detentor_registro = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet")
hist_dim_detentor_registro[['DETENTOR_REGISTRO','DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates().head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,1.0,ABACUS MEDICINE
1,ABBOLT,2.0,ABBOT
2,ABBOT,2.0,ABBOT
3,ABBOT BRASIL,2.0,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,2.0,ABBOT


In [42]:
hist_silver = fuzzy_merge(
    dim_df = hist_dim_detentor_registro,
    fact_df = hist_silver,
    dim_id_col="DETENTOR_REGISTRO_CHAVE",
    dim_text_col="DETENTOR_REGISTRO",
    fact_text_col="DETENTOR_REGISTRO",
    threshold=95, 
    suffix="" , 
    dedupe_on=False,
)

In [43]:
hist_silver['DETENTOR_REGISTRO_CHAVE'] = hist_silver['DETENTOR_REGISTRO_CHAVE'].fillna(0) 
hist_silver['DETENTOR_REGISTRO_CHAVE'].value_counts(dropna=False).head()

DETENTOR_REGISTRO_CHAVE
0      477515
224     11160
30       9325
84       7976
161      5470
Name: count, dtype: Int64

In [44]:
silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 24 columns):
 #   Column                                        Non-Null Count   Dtype 
---  ------                                        --------------   ----- 
 0   IDENTIFICACAO_NOTIFICACAO                     591457 non-null  object
 1   RELACAO_MEDICAMENTO_EVENTO                    590424 non-null  object
 2   NOME_MEDICAMENTO_WHODRUG                      586144 non-null  object
 3   PRINCIPIOS_ATIVOS_WHODRUG                     585614 non-null  object
 4   CODIGO_ATC                                    586144 non-null  object
 5   DETENTOR_REGISTRO                             138463 non-null  object
 6   CONCENTRACAO                                  100272 non-null  object
 7   COMPONENTE_SUSPEITO                           60890 non-null   object
 8   ACAO_ADOTADA                                  305550 non-null  object
 9   PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO  20166 non-nul

### ✅ FORMA_FARMACEUTICA

In [45]:
hist_dim_forma_farmaceutica = pd.read_parquet(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica.parquet")
hist_dim_forma_farmaceutica[['FORMA_FARMACEUTICA','FORMA_FARMACEUTICA_CHAVE', 'FORMA_FARMACEUTICA_VALOR']].drop_duplicates().head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,Adesivo Transdérmico,1,Adesivo transdermico
1,adesivo,1,Adesivo transdermico
2,adesivo transdérmico,1,Adesivo transdermico
3,Adesivo,1,Adesivo transdermico
5,ADESIVO,1,Adesivo transdermico


In [46]:
hist_silver.shape

(591457, 45)

In [47]:
hist_silver = fuzzy_merge(
    dim_df = hist_dim_forma_farmaceutica,
    fact_df = hist_silver,
    dim_id_col="FORMA_FARMACEUTICA_CHAVE",
    dim_text_col="FORMA_FARMACEUTICA",
    fact_text_col="FORMA_FARMACEUTICA",
    threshold=95, 
    suffix="" , 
    dedupe_on=False,
)

In [48]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 47 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                     591457 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                    590424 non-null  object        
 2   NOME_MEDICAMENTO_WHODRUG                      586144 non-null  object        
 3   PRINCIPIOS_ATIVOS_WHODRUG                     585614 non-null  object        
 4   CODIGO_ATC                                    586144 non-null  object        
 5   DETENTOR_REGISTRO                             113942 non-null  object        
 6   CONCENTRACAO                                  100272 non-null  object        
 7   COMPONENTE_SUSPEITO                           60890 non-null   object        
 8   ACAO_ADOTADA                                  305550 n

In [49]:
hist_silver['FORMA_FARMACEUTICA_CHAVE'] = hist_silver['FORMA_FARMACEUTICA_CHAVE'].fillna(0) 

In [50]:
hist_silver[['FORMA_FARMACEUTICA_CHAVE']].value_counts(dropna=False)

FORMA_FARMACEUTICA_CHAVE
0                           430359
14                           71146
4                            37369
12                           16150
16                           14393
11                           11338
2                             5318
10                            1501
15                            1489
3                              595
1                              345
20                             271
8                              246
19                             184
18                             155
13                             127
9                              122
6                              113
5                               94
7                               88
17                              54
Name: count, dtype: int64

### ✅ INDICACAO_MEDDRA

In [51]:
dim_soc_llt = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet")
dim_soc_llt["LLT_CHAVE"] = dim_soc_llt["LLT_CHAVE"].astype(int)  
dim_soc_llt.head()

,SOC_CHAVE,SOC_VALOR,HLGT,HLT,PT,LLT_CHAVE,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [52]:
hist_silver = fuzzy_merge(
    dim_df=dim_soc_llt[['LLT_CHAVE', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR']],
    fact_df=hist_silver,
    dim_id_col="LLT_CHAVE",
    dim_text_col="REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR",
    fact_text_col="INDICACAO_MEDDRA",
    threshold=98,
    suffix="_x"  # opcional; default "_MATCH"
)

In [53]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

### ✅ INDICACAO_RELATADA_NOTIFICADOR_INICIAL

In [54]:
hist_silver.INDICACAO_RELATADA_NOTIFICADOR_INICIAL.value_counts().head()

INDICACAO_RELATADA_NOTIFICADOR_INICIAL
PRODUCT USED FOR UNKNOWN INDICATION    12215
DRUG USE FOR UNKNOWN INDICATION         6457
DOENÇA DE CROHN                         3220
DIABETES                                3157
HIPERTENSÃO                             2975
Name: count, dtype: int64

In [55]:
hist_silver = fuzzy_merge(
    dim_df = dim_soc_llt[['LLT_CHAVE', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR']],
    fact_df = hist_silver,
    dim_id_col="LLT_CHAVE",
    dim_text_col="REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR",
    fact_text_col="INDICACAO_RELATADA_NOTIFICADOR_INICIAL",
    threshold = 98,
    suffix="_y")



In [56]:
hist_silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALOR_DENOMINADOR

## Pivot

### ✅ PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO

In [57]:
from utils import expandir_lista_wide

In [58]:
hist_silver = expandir_lista_wide(
    hist_silver,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)


In [59]:

hist_silver.filter(like="PROB_ADIC_").head(2)

,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0


In [60]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591457 entries, 0 to 591456
Data columns (total 64 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              591457 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                             590424 non-null  object        
 2   NOME_MEDICAMENTO_WHODRUG                               586144 non-null  object        
 3   PRINCIPIOS_ATIVOS_WHODRUG                              585614 non-null  object        
 4   CODIGO_ATC                                             586144 non-null  object        
 5   DETENTOR_REGISTRO                                      113942 non-null  object        
 6   CONCENTRACAO                                           100272 non-null  object        
 7   COMPONENTE_SUSPEITO                                    6

### ✅ CODIGO_ATC

In [61]:
from utils import desagrupar_colunas_pipe

In [62]:
colunas_para_desagrupar = ['ATC_CODE_LEVEL_4']
print(f"Shape antes da desagregação: {hist_silver.shape}") 
hist_silver_dedup = desagrupar_colunas_pipe(hist_silver.rename(columns={'CODIGO_ATC': 'ATC_CODE_LEVEL_4'}), colunas_para_desagrupar)
print(f"Shape depois da desagregação: {hist_silver_dedup.shape}") 

Shape antes da desagregação: (591457, 64)
Shape depois da desagregação: (804658, 64)


In [63]:
hist_silver_dedup["HASH_SILVER"] = build_row_hash(
    hist_silver_dedup, 
    hist_silver_dedup.columns.difference(['ATUALIZADO', 'HASH_BRONZE']).tolist(), 
    algo="sha256", 
    sep="|"
)

In [64]:
hist_silver_dedup.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG',
       'ATC_CODE_LEVEL_4', 'DETENTOR_REGISTRO', 'CONCENTRACAO',
       'COMPONENTE_SUSPEITO', 'ACAO_ADOTADA',
       'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 'INDICACAO_MEDDRA',
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE', 'FREQUENCIA_DOSE',
       'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO', 'FIM_ADMINISTRACAO',
       'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO', 'VIA_ADMINISTRACAO_MAE_PAI',
       'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALO

In [65]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/02_silver/hist_fat_medicamentos/hist_fat_medicamentos.parquet", index=False)

In [66]:
hist_silver_dedup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 804658 entries, 0 to 804657
Data columns (total 65 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO                              804658 non-null  object        
 1   RELACAO_MEDICAMENTO_EVENTO                             803063 non-null  object        
 2   NOME_MEDICAMENTO_WHODRUG                               804658 non-null  object        
 3   PRINCIPIOS_ATIVOS_WHODRUG                              803661 non-null  object        
 4   ATC_CODE_LEVEL_4                                       804658 non-null  object        
 5   DETENTOR_REGISTRO                                      138214 non-null  object        
 6   CONCENTRACAO                                           142804 non-null  object        
 7   COMPONENTE_SUSPEITO                                    8

# 🥇 Gold

In [67]:
hist_silver_dedup.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG',
       'ATC_CODE_LEVEL_4', 'DETENTOR_REGISTRO', 'CONCENTRACAO',
       'COMPONENTE_SUSPEITO', 'ACAO_ADOTADA',
       'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 'INDICACAO_MEDDRA',
       'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE', 'FREQUENCIA_DOSE',
       'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO', 'FIM_ADMINISTRACAO',
       'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO', 'VIA_ADMINISTRACAO_MAE_PAI',
       'NUMELO_LOTE', 'ATUALIZADO', 'HASH_BRONZE',
       'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
       'COMPONENTE_SUSPEITO_VALOR', 'COMPONENTE_SUSPEITO_CHAVE',
       'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE', 'DURACAO_TIPO_VALOR',
       'DURACAO_TIPO_CHAVE', 'DURACAO_VALOR', 'CONCENTRACAO_TIPO_VALOR',
       'CONCENTRACAO_TIPO_CHAVE', 'CONCENTRACAO_VALOR',
       'CONCENTRACAO_VALOR_NUMERADOR', 'CONCENTRACAO_VALO

In [68]:
# Lista base de colunas (use sempre os nomes de origem aqui)
gold_cols = [
    'IDENTIFICACAO_NOTIFICACAO', 
    
    #'RELACAO_MEDICAMENTO_EVENTO',
    'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
    'RELACAO_MEDICAMENTO_EVENTO_VALOR', 
    
    'NOME_MEDICAMENTO_WHODRUG', 
    'PRINCIPIOS_ATIVOS_WHODRUG',
    'ATC_CODE_LEVEL_4', 
    
    'DETENTOR_REGISTRO', 
    'DETENTOR_REGISTRO_SCORE', 
    'DETENTOR_REGISTRO_CHAVE',

    #'CONCENTRACAO',
    'CONCENTRACAO_TIPO_CHAVE', 
    'CONCENTRACAO_TIPO_VALOR',
    'CONCENTRACAO_VALOR',
    'CONCENTRACAO_VALOR_NUMERADOR', 
    'CONCENTRACAO_VALOR_DENOMINADOR',

    #'COMPONENTE_SUSPEITO',
    'COMPONENTE_SUSPEITO_CHAVE',
    'COMPONENTE_SUSPEITO_VALOR', 

    #'ACAO_ADOTADA',
    'ACAO_ADOTADA_CHAVE',
    'ACAO_ADOTADA_VALOR', 

    #'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
    'PROB_ADIC_ABUSO', 
    'PROB_ADIC_ERRO_DE_MEDICACAO', 'PROB_ADIC_EXPOSICAO_OCUPACIONAL','PROB_ADIC_FALSIFICACAO',
    'PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES',
    'PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES',
    'PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE',
    'PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI', 'PROB_ADIC_SUPERDOSE',
    'PROB_ADIC_USO_INCORRETO', 'PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO',
       
    #'INDICACAO_MEDDRA',
    #'INDICACAO_MEDDRA_SCORE', 
    'LLT_CHAVE_x', 
    #'INDICACAO_MEDDRA_x',
    
    
    #'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 
    #'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_SCORE', 
    'LLT_CHAVE_y',
    #'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_y', 
    

    #'DOSE', 
    'DOSE_TIPO_CHAVE', 
    'DOSE_TIPO_VALOR', 
    'DOSE_VALOR',
    
    'FREQUENCIA_DOSE',
    
    'POSOLOGIA', 
    
    #'DURACAO', 
    'DURACAO_TIPO_CHAVE', 
    'DURACAO_TIPO_VALOR',
    'DURACAO_VALOR', 
    
    'INICIO_ADMINISTRACAO', 
    'FIM_ADMINISTRACAO',
    
    #'FORMA_FARMACEUTICA', 
    #'FORMA_FARMACEUTICA_SCORE', 
    'FORMA_FARMACEUTICA_CHAVE',
    
    #'VIA_ADMINISTRACAO', 
    'VIA_ADMINISTRACAO_CHAVE',
    
    #'VIA_ADMINISTRACAO_MAE_PAI',
    'VIA_ADMINISTRACAO_MAE_PAI_CHAVE',
    
    'NUMELO_LOTE', 
    'ATUALIZADO', 
    'HASH_BRONZE',
    'HASH_SILVER'
]

# Dicionário de renomes: chave = nome original, valor = novo nome
rename_map = {
    'LLT_CHAVE_x': 'INDICACAO_MEDDRA_LLT_CHAVE',
    'LLT_CHAVE_y': 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_LLT_CHAVE',
    # adicione outras renomeações aqui, ex:
    # 'NUMELO_LOTE': 'NUMERO_LOTE',
}

# Lista final de colunas já com renomeação aplicada
final_cols = [rename_map.get(col, col) for col in gold_cols]

# Opcional: se quiser aplicar no DataFrame pandas
fat = hist_silver_dedup.rename(columns=rename_map)
fat = fat[final_cols]

In [69]:
fat["HASH_GOLD"] = build_row_hash(fat, final_cols, algo="sha256", sep="|")

In [70]:
fat.head()

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,DETENTOR_REGISTRO,DETENTOR_REGISTRO_SCORE,DETENTOR_REGISTRO_CHAVE,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR,CONCENTRACAO_VALOR,CONCENTRACAO_VALOR_NUMERADOR,CONCENTRACAO_VALOR_DENOMINADOR,COMPONENTE_SUSPEITO_CHAVE,COMPONENTE_SUSPEITO_VALOR,ACAO_ADOTADA_CHAVE,ACAO_ADOTADA_VALOR,PROB_ADIC_ABUSO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO,INDICACAO_MEDDRA_LLT_CHAVE,INDICACAO_RELATADA_NOTIFICADOR_INICIAL_LLT_CHAVE,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR,DOSE_VALOR,FREQUENCIA_DOSE,POSOLOGIA,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA_CHAVE,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_MAE_PAI_CHAVE,NUMELO_LOTE,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300000004,1,SUSPEITO,OXACILINA,OXACILLIN SODIUM,J01CF,None,90.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,0,DESCONHECIDO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),250.0,6 HORA,None,4,DIA,4.0,2018-11-24,NaT,0,6,5,5833018,2026-01-15,3E94C6FE0A6106B170D0EA675DCBCDDEAE7B2A96987C79E898FDB29A3C68E0CC,d172d53e618076b054e9580d6af3b9942f7393b0012576231c65badc227f2ceb,6dc7d7b92819df4aa149222f238164d1e67b7844ce060c6398195b7daf4743d7
1,BR-ANVISA-300000005,1,SUSPEITO,PARACEMOL,PARACETAMOL,N02BE,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,0,DESCONHECIDO,NaN,None,None,0,DESCONHECIDO,NaN,2018-11-22,2018-11-22,0,5,5,None,2026-01-15,845CF3FFA604507A5F50CBFF0488491FF396AE79C33B0D22D9C495A5CE3E3274,5017481c9bafddbf3c4b1ffdb8765fb331a0a77f90288eb7c6565524349bb107,7f2dd2122b0b91bf3e59e836aa54129706f71d66fd693dd132050a74ea426835
2,BR-ANVISA-300000007,1,SUSPEITO,DIAMOX,ACETAZOLAMIDE SODIUM,C03,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),250.0,6 HORA,250MG A CADA 6 HORAS,0,DESCONHECIDO,NaN,2018-11-03,2018-11-15,0,5,5,None,2026-01-15,C2A772E01917C72263CBE3D823B6F0C330BFDDCD2FF70ABBDB30FE3D8887EA6F,aa7fc00f4d2b09192b345d034a7b0fcec0f96d0e61054118be3ff3b9f6360389,88b550c0c5f20dbf488b18eb19a6534e97549381e0ff8ce5e24fe8738d673d90
3,BR-ANVISA-300000007,1,SUSPEITO,DIAMOX,ACETAZOLAMIDE SODIUM,N03AX,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),250.0,6 HORA,250MG A CADA 6 HORAS,0,DESCONHECIDO,NaN,2018-11-03,2018-11-15,0,5,5,None,2026-01-15,C2A772E01917C72263CBE3D823B6F0C330BFDDCD2FF70ABBDB30FE3D8887EA6F,3259b0be313c4e4e0673af8b672360255351eff24acc429456806d2c6b7bb8ae,48ed26c1d4489b254ecf29b89b91fa5ba18a9da5e0628ebada10d5bc2d827456
4,BR-ANVISA-300000007,1,SUSPEITO,DIAMOX,ACETAZOLAMIDE SODIUM,S01EC,None,0.0,0,0,DESCONHECIDO,NaN,NaN,NaN,0,DESCONHECIDO,1,RETIRADA DO MEDICAMENTO,0,0,0,0,0,0,0,0,0,0,0,0,0,1,MILLIGRAM (MG),250.0,6 HORA,250MG A CADA 6 HORAS,0,DESCONHECIDO,NaN,2018-11-03,2018-11-15,0,5,5,None,2026-01-15,C2A772E01917C72263CBE3D823B6F0C330BFDDCD2FF70ABBDB30FE3D8887EA6F,5dad67417f1784dda3dc70466f650943fbad4eec07d14d7399ead6925b38fd3c,d716d0e66cd7ea8867bc8f1d19dc37755e1bc09f37ff0961d85c4524ce0808dd


In [71]:
fat.to_parquet(Path(project_root) / "data/03_gold/fat_medicamentos/fat_medicamentos.parquet", index=False)